# 🧠 Hermes Skills Lab: Crafting Agentic Workflows with Local & GPU-Hosted LLMs

**Workshop Notebook 1 of 2**

Welcome! In this notebook you will:
1. Install Hermes Agent from scratch — no prior setup needed
2. Install **Ollama** and run a **local**, lightweight LLM (Llama 3.2 1B) on your own laptop — no API keys required
3. Learn how to write **Skills** — the building blocks of Hermes workflows
4. Build a real-world use-case — 📧 **Gmail Spam Detector** — first against the local model, then reconfigure Hermes to point at an **8B model running on a remote L4 GPU** for the live inbox test

Parts 1–3 run entirely on your laptop so you can see, first-hand, where a tiny local model hits its ceiling. Part 5 then switches Hermes over to real GPU compute for the actual live demo.

---
> **Parts 1–4 need no GPU or cloud account** — everything runs locally via Ollama. Part 5 connects to a GPU VM you control.
---

## 📋 Prerequisites

You need:
- **~2GB free disk space** (Ollama itself + a small local model, roughly 1.3GB)
- **Homebrew** (macOS, recommended) → https://brew.sh — used to install Ollama. Prefer not to use Homebrew? Install Ollama manually from https://ollama.com/download/mac instead
- *(Optional — only needed for the live Gmail test)* A Gmail App Password → https://myaccount.google.com/apppasswords
- **For Part 5:** an L4 GPU VM running the vLLM OpenAI-compatible server (setup covered separately), reachable at a URL/port you control, plus the API key you configured for it

**Operating System:** macOS (this notebook is tuned for MacBook Air), Linux, or WSL2 on Windows

---
## Part 1 — Environment Setup & Installing Hermes Agent

Hermes Agent needs Python, Node.js, and a few CLI tools. The installer handles everything automatically.

In [26]:
# Step 1: Check your Python version (needs 3.9+)
import sys
print(f"Python version: {sys.version}")
assert sys.version_info >= (3, 9), "Please upgrade to Python 3.9 or higher"

Python version: 3.12.13 (main, Mar  3 2026, 12:39:30) [Clang 17.0.0 (clang-1700.6.4.2)]


In [27]:
# Step 2: Check if git is available (required by installer)
import subprocess

result = subprocess.run(["git", "--version"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"✅ Git found: {result.stdout.strip()}")
else:
    print("❌ Git not found. Install it:")
    print("  Ubuntu/Debian: sudo apt install git")
    print("  macOS:         brew install git")

✅ Git found: git version 2.50.1 (Apple Git-155)


In [28]:
# Step 3: Install Hermes Agent
# This one command installs Python 3.11, Node.js v22, ripgrep, ffmpeg, and Hermes itself.
# Skips straight past the (slow) reinstall if Hermes is already present — only a
# first-time install or a forced reinstall pays the dependency-resolution cost.

import subprocess, os, shutil

FORCE_REINSTALL = False   # flip to True and re-run this cell to force a clean reinstall/update

home = os.path.expanduser("~")
local_bin = os.path.join(home, ".local", "bin")
hermes_path = os.path.join(local_bin, "hermes")

check_env = os.environ.copy()
check_env["PATH"] = local_bin + os.pathsep + check_env.get("PATH", "")

def hermes_already_installed():
    if not (shutil.which("hermes", path=check_env["PATH"]) or os.path.exists(hermes_path)):
        return False, None
    result = subprocess.run(["hermes", "--version"], capture_output=True, text=True, env=check_env, timeout=15)
    if result.returncode == 0:
        return True, result.stdout.strip()
    return False, None

already_installed, version_info = (False, None) if FORCE_REINSTALL else hermes_already_installed()

if already_installed:
    print("✅ Hermes Agent already installed — skipping install\n")
    print(version_info)
    print("\n(Set FORCE_REINSTALL = True above and re-run this cell to force a clean reinstall/update.)")
else:
    INSTALL_CMD = "curl -fsSL https://hermes-agent.nousresearch.com/install.sh | bash"

    # On networks with a TLS-inspecting proxy/VPN/antivirus (common on corporate or
    # school Wi-Fi), the installer's package manager (uv) can fail with:
    #   "invalid peer certificate: UnknownIssuer"
    # because uv ships its own certificate bundle instead of trusting the OS's.
    #
    #  - UV_SYSTEM_CERTS tells uv to trust the same certificate store your Mac
    #    already trusts (fixes the issue on most machines). This replaced the
    #    older UV_NATIVE_TLS variable, which newer uv versions warn is deprecated.
    #  - On some macOS setups, uv's system-certificate detection doesn't pick up
    #    the interception CA even though the OS itself trusts it (a known uv
    #    limitation). UV_INSECURE_HOST is a scoped fallback that skips certificate
    #    verification ONLY for the two PyPI hosts Hermes needs — not a global TLS
    #    bypass. Safe on a trusted workshop/home network; avoid on public Wi-Fi.
    env = os.environ.copy()
    env["UV_SYSTEM_CERTS"] = "1"
    env["UV_INSECURE_HOST"] = "pypi.org files.pythonhosted.org"

    print("Installing Hermes Agent... (this may take ~90 seconds)")
    print("="*60)

    proc = subprocess.Popen(
        INSTALL_CMD, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, env=env, bufsize=1,
    )
    output_lines = []
    for line in proc.stdout:
        print(line, end="")
        output_lines.append(line)
    proc.wait()
    output = "".join(output_lines)

    if proc.returncode == 0:
        print("\n✅ Hermes Agent installed successfully!")
    else:
        print(f"\n❌ Installation failed with code {proc.returncode}")
        if "UnknownIssuer" in output:
            print("Still seeing a certificate error even with the fallbacks above enabled.")
            print("Run this manually in Terminal and share the output if it still fails:")
        else:
            print("Try running manually in your terminal:")
        print("  export UV_SYSTEM_CERTS=1 UV_INSECURE_HOST='pypi.org files.pythonhosted.org'")
        print("  curl -fsSL https://hermes-agent.nousresearch.com/install.sh | bash")

✅ Hermes Agent already installed — skipping install

Hermes Agent v0.18.2 (2026.7.7.2) · upstream e4ea0a0e
Install directory: /Users/mohanrajdavala/.hermes/hermes-agent
Install method: git
Python: 3.11.15
OpenAI SDK: 2.24.0

(Set FORCE_REINSTALL = True above and re-run this cell to force a clean reinstall/update.)


In [29]:
# Step 4: Reload PATH so the 'hermes' command is available in this notebook
import os

home = os.path.expanduser("~")
local_bin = os.path.join(home, ".local", "bin")

if local_bin not in os.environ["PATH"]:
    os.environ["PATH"] = local_bin + ":" + os.environ["PATH"]
    print(f"✅ Added {local_bin} to PATH")
else:
    print(f"✅ {local_bin} already in PATH")

# Verify hermes is available
result = subprocess.run(["hermes", "--version"], capture_output=True, text=True, env=os.environ)
if result.returncode == 0:
    print(f"\n🎉 Hermes version: {result.stdout.strip()}")
else:
    # Try the full path
    hermes_path = os.path.join(home, ".hermes", "hermes-agent", "venv", "bin", "hermes")
    if os.path.exists(hermes_path):
        os.environ["HERMES"] = hermes_path
        print(f"✅ Hermes found at: {hermes_path}")
    else:
        print("❌ hermes command not found. Please restart the kernel after installation.")

✅ /Users/mohanrajdavala/.local/bin already in PATH

🎉 Hermes version: Hermes Agent v0.18.2 (2026.7.7.2) · upstream e4ea0a0e
Install directory: /Users/mohanrajdavala/.hermes/hermes-agent
Install method: git
Python: 3.11.15
OpenAI SDK: 2.24.0


In [30]:
# Helper function we'll use throughout this notebook
def run_hermes(args, input_text=None, timeout=60):
    """Run a hermes command and return the output."""
    cmd = ["hermes"] + args
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        input=input_text,
        timeout=timeout,
        env=os.environ
    )
    return result.stdout + result.stderr

print("✅ Helper function ready")

✅ Helper function ready


---
## Part 2 — Install Ollama & Run a Local Model

### Why Llama 3.2 1B?

A MacBook Air has no discrete GPU, so an 8B+ model is too heavy for its unified memory to run comfortably. [Ollama](https://ollama.com/) solves this by running fully **on-device** — no API keys, no rate limits, no internet dependency once the model is downloaded. Getting here took ruling out two other options, each for a different reason:

1. **Qwen3 0.6B (sub-1B)** — Hermes requires **at least 64,000 tokens of context** for agentic/tool-calling use. Essentially no model under 1B parameters is trained with a native context window that large (they're almost all capped around 32K), and Ollama silently clamps your requested context down to whatever the model actually supports (you'll see `"only 40,960 tokens of runtime context"`-style errors if you try).
2. **Llama 3.2 3B** — clears the context floor and is Ollama/Meta's tool-use-tagged variant, but hit a separate, confirmed Ollama bug: `llama3.2:3b` tool calls sometimes land as plain JSON text in the response `content` field instead of the structured `tool_calls` field the OpenAI-compatible endpoint is supposed to use — so Hermes never actually executes them ([ollama/ollama#13519](https://github.com/ollama/ollama/issues/13519)).

**Llama 3.2 1B** is what we're using here — smallest and fastest of the three, 128K native context clears Hermes's floor with no config tricks. The trade-off: Ollama doesn't even list "tool use" as one of its capabilities (unlike the 3B), so expect it to often *describe* a shell command instead of actually invoking it, or hallucinate a plausible-looking answer entirely. That's a real, demonstrated limitation, not a bug to keep chasing — it's the "small model ceiling" this notebook is designed to show you before Notebook 2 hands the same tasks to a 35B model.

| Model | Params | Ollama tag | Native context | Tool Calling | Notes |
|-------|--------|-----------|-----------------|-------------|-------|
| **Llama 3.2 1B** | 1.24B | `llama3.2:1b` | **128K** | ⭐⭐ | **Used in this notebook** — smallest/fastest option; not tool-use-tagged by Ollama, so expect unreliable tool calls |
| Llama 3.2 3B | ~3.2B | `llama3.2:3b` | 128K | ⭐⭐⭐⭐ (when it works) | Ollama/Meta's tool-use-tagged variant, but hits a confirmed OpenAI-compat tool-call parsing bug in some Ollama versions |
| Qwen3 0.6B | 0.6B | `qwen3:0.6b` | ~40K (Ollama-packaged) | ⭐⭐⭐ | Genuinely sub-1B, but its context ceiling sits *below* what Hermes requires — needs a custom Modelfile workaround, not guaranteed to work |
| Qwen2.5 0.5B | 0.5B | `qwen2.5:0.5b` | ~32K | ⭐⭐ | Same context ceiling problem as Qwen3 0.6B |

> ⚠️ Small local models are noticeably less capable than Qwen3-8B or the 35B model in Notebook 2 — expect them to struggle with multi-step tool use and complex instructions, including sometimes failing to call a tool at all. That's the point: you'll feel the ceiling first-hand before Notebook 2 shows what a bigger model unlocks.

> **Hermes requires ≥64,000 tokens of context** for agentic/tool-calling use. Ollama also defaults to a much smaller context window on machines with under 24GB of VRAM/unified memory, so the cells below explicitly set `OLLAMA_CONTEXT_LENGTH=64000` — safe here since Llama 3.2 1B's native 128K context comfortably covers it.

In [31]:
# Step 1: Install (or update) Ollama
# Tool-calling bugs in Ollama's OpenAI-compat layer get fixed over time, so if
# Ollama is already installed we still run a quick `brew upgrade` to pick up
# fixes — this is fast/idempotent, unlike the multi-minute Hermes reinstall.
import platform, shutil, subprocess, os, urllib.request

os_name = platform.system()
print(f"Detected OS: {os_name}")

if shutil.which("ollama"):
    print("✅ Ollama already installed")
    if os_name == "Darwin" and shutil.which("brew"):
        print("Checking for updates via Homebrew...")
        result = subprocess.run(["brew", "upgrade", "ollama"], capture_output=True, text=True)
        out = (result.stdout + result.stderr).strip()
        if "already installed" in out.lower() or not out:
            print("   Already on the latest Homebrew version.")
        else:
            print(out[-2000:])
            # If we just upgraded the binary, any already-running `ollama serve`
            # process is still the OLD binary — stop it so Step 2 starts fresh.
            try:
                urllib.request.urlopen("http://localhost:11434", timeout=2)
                print("   Restarting the running Ollama server to pick up the update...")
                subprocess.run(["pkill", "-f", "ollama serve"], capture_output=True)
            except Exception:
                pass
else:
    if os_name == "Darwin":
        if shutil.which("brew"):
            print("Installing Ollama via Homebrew... (this may take a minute)")
            result = subprocess.run(["brew", "install", "ollama"], capture_output=True, text=True)
            print(result.stdout[-2000:])
            if result.returncode != 0:
                print(result.stderr[-2000:])
        else:
            print("❌ Homebrew not found.")
            print("   Install Homebrew (https://brew.sh) and re-run this cell, OR")
            print("   download the Ollama app directly: https://ollama.com/download/mac")
    elif os_name == "Linux":
        print("Installing Ollama... (this may take ~60 seconds)")
        subprocess.run(
            "curl -fsSL https://ollama.com/install.sh | sh",
            shell=True, capture_output=False, text=True
        )
    else:
        print(f"⚠️  Unsupported OS for auto-install: {os_name}")
        print("   Download manually from https://ollama.com/download")

if shutil.which("ollama"):
    # `ollama --version` also tries to contact a running server and prints a
    # scary-looking "could not connect to a running Ollama instance" warning if
    # it's not up yet — that's expected at this point, so we only surface the
    # client version here. The next cell (Step 2) starts the server.
    result = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
    client_line = next(
        (l for l in result.stdout.splitlines() if "version" in l.lower()),
        result.stdout.strip(),
    )
    print(f"\n🎉 Ollama ready — {client_line}")
    print("   Run the next cell (Step 2) to (re)start the server.")
else:
    print("❌ 'ollama' command not found — restart the kernel after installing, then re-run this cell")

Detected OS: Darwin
✅ Ollama already installed
Checking for updates via Homebrew...
   Already on the latest Homebrew version.

🎉 Ollama ready — ollama version is 0.32.0
   Run the next cell (Step 2) to (re)start the server.


In [32]:
# Step 2: Start the Ollama server in the background with a 64K context window
# (Hermes requires >=64K tokens of context for tool-calling agents — see note above)
import subprocess, os, time, urllib.request

def ollama_is_running():
    try:
        urllib.request.urlopen("http://localhost:11434", timeout=2)
        return True
    except Exception:
        return False

env = os.environ.copy()
env["OLLAMA_CONTEXT_LENGTH"] = "64000"

if ollama_is_running():
    print("✅ Ollama server already running on port 11434")
else:
    print("Starting `ollama serve` in the background with 64K context...")
    log_dir = os.path.expanduser("~/.hermes")
    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, "ollama_serve.log")
    log_file = open(log_path, "w")
    subprocess.Popen(["ollama", "serve"], env=env, stdout=log_file, stderr=log_file)

    for _ in range(20):
        if ollama_is_running():
            print("✅ Ollama server is up (http://localhost:11434)")
            break
        time.sleep(1)
    else:
        print(f"⚠️  Server didn't respond after 20s — check the log at: {log_path}")

✅ Ollama server already running on port 11434


In [33]:
# Step 3: Pull the small local model — Llama 3.2 1B (~1.3GB, one-time download)
import subprocess

OLLAMA_MODEL = "llama3.2:1b"   # Smallest/fastest option with 128K native context; tool-calling is unreliable (expected)

print(f"Pulling {OLLAMA_MODEL}...")
result = subprocess.run(["ollama", "pull", OLLAMA_MODEL], capture_output=False, text=True)

if result.returncode == 0:
    print(f"\n✅ {OLLAMA_MODEL} ready")
else:
    print(f"\n❌ Pull failed with code {result.returncode}")

subprocess.run(["ollama", "list"])

Pulling llama3.2:1b...


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling 74701a8c35f6: 100% ▕██████████████████▏ 1.3 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 4f659a1e86d7: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 



✅ llama3.2:1b ready
NAME                       ID              SIZE      MODIFIED               
llama3.2:1b                baf6a787fdff    1.3 GB    Less than a second ago    
nomic-embed-text:latest    0a109f422b47    274 MB    4 months ago              


CompletedProcess(args=['ollama', 'list'], returncode=0)

In [34]:
# Step 4: Point Hermes at the local Ollama model
#
# NOTE: this uses a NAMED provider entry under "providers:", not the inline
# "model: {provider: custom, base_url: ...}" shorthand. The inline path has a
# confirmed, open Hermes Agent bug (github.com/NousResearch/hermes-agent/issues/43586)
# around auth resolution. Ollama itself doesn't require an API key, but this
# same ~/.hermes/config.yaml file is shared with Notebook 2 and Part 4b below,
# both of which add their own named GPU provider entries (h100vllm, l4vllm) to
# "providers:". A non-empty "providers:" dict alongside the inline "custom"
# shorthand is exactly the kind of state that resolver bug can misroute on -
# so we use the named-provider path everywhere, which does not have this bug.
import os, yaml

hermes_dir = os.path.expanduser("~/.hermes")
os.makedirs(hermes_dir, exist_ok=True)

config_path = os.path.join(hermes_dir, "config.yaml")

if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config = yaml.safe_load(f) or {}
else:
    config = {}

PROVIDER_NAME = "localollama"

if "providers" not in config or not isinstance(config["providers"], dict):
    config["providers"] = {}

config["model"] = {
    "default": OLLAMA_MODEL,
    "provider": PROVIDER_NAME,
    "context_length": 64000,   # Hermes requires >=64K context for tool-calling
}

config["providers"][PROVIDER_NAME] = {
    "name": PROVIDER_NAME,
    "base_url": "http://localhost:11434/v1",
    "default_model": OLLAMA_MODEL,
    "api_mode": "chat_completions",
    # No key_env - Ollama's local server does not require authentication.
}

with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"✅ Config written to {config_path}")
print(f"   Model:    {OLLAMA_MODEL}")
print(f"   Provider: '{PROVIDER_NAME}' (local Ollama — no API key required)")
print(f"   Endpoint: http://localhost:11434/v1")

✅ Config written to /Users/mohanrajdavala/.hermes/config.yaml
   Model:    llama3.2:1b
   Provider: 'localollama' (local Ollama — no API key required)
   Endpoint: http://localhost:11434/v1


In [35]:
# Verify Hermes can read the configuration
output = run_hermes(["doctor"])
print(output[:3000])  # Show first 3000 chars


┌─────────────────────────────────────────────────────────┐
│                 🩺 Hermes Doctor                        │
└─────────────────────────────────────────────────────────┘

◆ Security Advisories
  ✓ No active security advisories

◆ MCP Server Security
  ✓ No suspicious MCP stdio commands

◆ Python Environment
  ✓ Python 3.11.15
  ✓ Virtual environment active
  ✓ Version files consistent (0.18.2)

◆ SSL / CA Certificates
  ✓ SSL CA certificate bundle is valid

◆ Required Packages
  ✓ OpenAI SDK
  ✓ Rich (terminal UI)
  ✓ python-dotenv
  ✓ PyYAML
  ✓ HTTPX
  ✓ Croniter (cron expressions) (optional)
  ✓ python-telegram-bot (optional)
  ⚠ discord.py (optional, not installed)

◆ Configuration Files
  ✓ ~/.hermes/.env file exists
  ✓ API key or custom endpoint configured
  ✓ ~/.hermes/config.yaml exists
  ⚠ Config version outdated (v30 → v33) (new settings available)

◆ xAI Model Retirement (May 15, 2026)
  ✓ No retired xAI models in config

◆ Auth Providers
  ⚠ Nous Portal auth (not

---
## Part 3 — Your First Hermes Conversation

Hermes runs as a CLI by default. From a Jupyter notebook, we send it a one-shot prompt using the `-z` flag, which prints just the final response and exits — nothing else on stdout/stderr (great for demos). Note: older Hermes versions used `--print` for this; current versions use `-z`.

In [36]:
# Safety check: make sure Hermes is actually pointing at your LOCAL Ollama
# model before running Part 3.
#
# ~/.hermes/config.yaml is a SINGLE SHARED file on disk. If you've already
# run Notebook 2 (H100 Qwen2.5-32B-Instruct) or Part 4b further down in this
# notebook (L4 GPU 8B model), those cells add NAMED provider entries
# (h100vllm, l4vllm) under "providers:" in that same file - and they persist
# even after you come back and re-run Part 3 in a new kernel.
#
# We ALSO switch Ollama itself to a named "providers:" entry here (instead of
# the inline "model: {provider: custom, base_url: ...}" shorthand). Reason:
# the inline shorthand has a confirmed Hermes Agent bug around auth
# resolution (github.com/NousResearch/hermes-agent/issues/43586), and in
# testing, a non-empty "providers:" dict (left behind by Notebook 2 / Part 4b)
# alongside that inline shorthand caused Hermes to misroute requests to a
# leftover GPU endpoint instead of localhost - producing the exact
# "HTTP 401: Unauthorized" you saw, even though Ollama itself needs no key.
# The named-provider path does not have this bug, so we use it everywhere.
#
# This cell unconditionally: (1) removes any stale GPU provider entries so
# there's nothing left for the resolver to get confused by, (2) writes a
# clean "localollama" named provider, and (3) prints exactly what Hermes
# itself resolves via `hermes config show`, so if this ever misbehaves again
# you can see the real state instead of guessing.
import os, yaml, subprocess

hermes_dir = os.path.expanduser("~/.hermes")
os.makedirs(hermes_dir, exist_ok=True)
config_path = os.path.join(hermes_dir, "config.yaml")

if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config = yaml.safe_load(f) or {}
else:
    config = {}

old_model = config.get("model", "unknown")

if "providers" not in config or not isinstance(config["providers"], dict):
    config["providers"] = {}

removed = [name for name in ("h100vllm", "l4vllm") if name in config["providers"]]
for stale in removed:
    del config["providers"][stale]
if removed:
    print(f"⚠️  Removed stale GPU provider entries: {removed}")

PROVIDER_NAME = "localollama"

config["model"] = {
    "default": OLLAMA_MODEL,
    "provider": PROVIDER_NAME,
    "context_length": 64000,   # Hermes requires >=64K context for tool-calling
}

config["providers"][PROVIDER_NAME] = {
    "name": PROVIDER_NAME,
    "base_url": "http://localhost:11434/v1",
    "default_model": OLLAMA_MODEL,
    "api_mode": "chat_completions",
    # No key_env - Ollama's local server does not require authentication.
}

with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"✅ Hermes is pointed at local Ollama for Part 3")
print(f"   Before: {old_model}")
print(f"   Model:    {OLLAMA_MODEL}")
print(f"   Provider: '{PROVIDER_NAME}' (local Ollama — no API key required)")
print(f"   Endpoint: http://localhost:11434/v1")

# Diagnostic: show exactly what Hermes itself thinks is configured right now.
diag = subprocess.run(["hermes", "config", "show"], capture_output=True, text=True, env=os.environ)
print("\n--- hermes config show (model/providers section) ---")
for line in (diag.stdout or diag.stderr).splitlines():
    low = line.strip().lower()
    if low.startswith("model") or low.startswith("providers") or "base_url" in low or "key_env" in low or "api_key" in low or "provider" in low:
        print(line)

⚠️  Removed stale GPU provider entries: ['h100vllm', 'l4vllm']
✅ Hermes is pointed at local Ollama for Part 3
   Before: {'context_length': 64000, 'default': 'llama3.2:1b', 'provider': 'localollama'}
   Model:    llama3.2:1b
   Provider: 'localollama' (local Ollama — no API key required)
   Endpoint: http://localhost:11434/v1

--- hermes config show (model/providers section) ---
  Model:        {'context_length': 64000, 'default': 'llama3.2:1b', 'provider': 'localollama'}
  Model:        (auto)


In [37]:
# Send a simple one-shot prompt to Hermes
# -z mode: run once, print the final response, exit

result = subprocess.run(
    ["hermes", "-z", "What is 15 * 37? Show your working."],
    capture_output=True, text=True, timeout=60, env=os.environ
)

print("=== HERMES RESPONSE ===")
print(result.stdout or result.stderr)

=== HERMES RESPONSE ===
To find 15 * 37, we can simply multiply these two numbers together.

15 × 40 = 600
Adding 7 to this product gives us the desired result:
600 + 7 = 607



In [38]:
# A slightly harder prompt — test tool use (listing files)
result = subprocess.run(
    ["hermes", "-z", "List the files in my home directory and tell me how many there are."],
    capture_output=True, text=True, timeout=90, env=os.environ
)

print("=== HERMES RESPONSE ===")
print(result.stdout or result.stderr)

=== HERMES RESPONSE ===
{"status": "ok", "profile": "automated", "output": {"files": "[ '/home/mohanrajdavala/Documents' ]"}}



In [39]:
# Free up memory on your MacBook by stopping the running Ollama model
#
# `ollama stop <model>` unloads the model from memory (RAM) without killing
# the Ollama server itself - the server process stays up, so you can reload
# the model later (e.g. `ollama run llama3.2:1b`) if you want to come back
# to it. This just kills the currently-running model process, freeing up
# the RAM it was using now that Parts 1-3 are done with it.
import subprocess

check = subprocess.run(["ollama", "ps"], capture_output=True, text=True)
print("Currently loaded Ollama models:")
print(check.stdout.strip() or "(none)")

if OLLAMA_MODEL in (check.stdout or ""):
    stop = subprocess.run(["ollama", "stop", OLLAMA_MODEL], capture_output=True, text=True)
    if stop.returncode == 0:
        print(f"\n✅ Stopped {OLLAMA_MODEL} — memory freed on your MacBook")
        print("   (Ollama's server is still running in the background — only the")
        print(f"   loaded model was unloaded. Run `ollama run {OLLAMA_MODEL}` any")
        print("   time to load it back into memory.)")
    else:
        print(f"\n⚠️  Could not stop {OLLAMA_MODEL}: {stop.stderr.strip()}")
else:
    print(f"\nℹ️  {OLLAMA_MODEL} isn't currently loaded into memory — nothing to stop")

Currently loaded Ollama models:
NAME           ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
llama3.2:1b    baf6a787fdff    3.5 GB    100% GPU     64000      4 minutes from now

✅ Stopped llama3.2:1b — memory freed on your MacBook
   (Ollama's server is still running in the background — only the
   loaded model was unloaded. Run `ollama run llama3.2:1b` any
   time to load it back into memory.)


---
## Part 4a — Understanding Skills

### What is a Skill?

A **Skill** in Hermes is a plain text file (`SKILL.md`) that teaches the agent how to do a specific task. Think of it like a recipe card:

```
~/.hermes/skills/
├── spam-detector/
│   └── SKILL.md       ← your skill
├── code-reviewer/
│   └── SKILL.md
└── ...                ← bundled Hermes skills
```

### Anatomy of a SKILL.md

```markdown
---
name: my-skill
description: One sentence — what does this skill do? (used for matching)
---

# My Skill Name

Brief explanation of the skill's purpose.

## Steps

1. Step one: what to do first
2. Step two: ...
3. ...

## Rules

- Always do X
- Never do Y
```

Hermes reads the `description` line cheaply on every request. It only loads the full SKILL.md content when a task actually matches. This means **you can have hundreds of skills without slowing things down**.

Skills become slash commands automatically:
```
/spam-detector check my inbox
```

In [40]:
# Let's write a simple practice skill first — a code reviewer
import os

skills_dir = os.path.expanduser("~/.hermes/skills")
os.makedirs(skills_dir, exist_ok=True)

practice_skill_dir = os.path.join(skills_dir, "code-reviewer")
os.makedirs(practice_skill_dir, exist_ok=True)

practice_skill = """---
name: code-reviewer
description: Review Python code for bugs, style issues, and suggest improvements
---

# Code Reviewer Skill

You are a Python code reviewer. When asked to review code, you carefully analyze it
and provide structured, actionable feedback.

## Review Checklist

1. **Correctness** — Does the code do what it claims? Are there logic bugs?
2. **Style** — Does it follow PEP 8? Are variable names clear?
3. **Edge cases** — What happens with empty inputs, None, or very large values?
4. **Performance** — Any obvious inefficiencies?
5. **Security** — Any hardcoded secrets or injection risks?

## Output Format

Always structure your review as:
- **Summary**: One sentence overall verdict
- **Issues Found**: Numbered list of problems (critical → minor)
- **Suggested Fixes**: Code snippets where helpful
- **Score**: X/10 with brief justification

## Rules

- Be constructive, not harsh
- Always show a corrected version for critical bugs
- If the code is good, say so clearly
"""

# Write file with UTF-8 encoding for Windows
# skill_file_path = os.path.join(practice_skill_dir, "SKILL.md")
# with open(skill_file_path, "w", encoding="utf-8") as f:
#     f.write(practice_skill)

# For MAC/Linux below works....
with open(os.path.join(practice_skill_dir, "SKILL.md"), "w") as f:
    f.write(practice_skill)

print("✅ Practice skill created at:", os.path.join(practice_skill_dir, "SKILL.md"))

✅ Practice skill created at: /Users/mohanrajdavala/.hermes/skills/code-reviewer/SKILL.md


## Step 4b: Point Hermes at Your L4 GPU

Up to now Hermes has been running against the local Llama 3.2 1B model on your MacBook Air — small, fast, and (as Part 3 showed) not reliable at tool calling. For this actual use-case, we reconfigure Hermes to talk to a real model server instead: an 8B model (`NousResearch/Hermes-3-Llama-3.1-8B`, run via [vLLM](https://docs.vllm.ai/)'s OpenAI-compatible API) on an L4 GPU VM.

**Before running the next cells, make sure the vLLM server is up on your L4 VM** — see the separate GPU setup doc for the `docker run` command, firewall rule, and API key generation.

Ollama keeps running locally in the background; nothing to shut down. We're just changing Hermes's `provider: custom` config to point at a different `base_url` — you can switch back to the local model any time by re-running Part 2's Step 4 cell.

In [41]:
# ============================================================
# 🔑 FILL IN YOUR L4 GPU VLLM ENDPOINT DETAILS
# ============================================================

GPU_BASE_URL = "http://103.42.51.73:1027/v1"
GPU_MODEL    = "hermes-3-llama-3.1-8b"
GPU_API_KEY  = "6da0856ebacbc687c811e4f86b29f4bb48924838e5e11468eec95f3fe9e34637"

# ============================================================

if GPU_API_KEY == "6da0856ebacbc687c811e4f86b29f4bb48924838e5e11468eec95f3fe9e34637":
    print("GPU_API_KEY = {}".format(GPU_API_KEY))
else:
    print("GPU endpoint details set")
    print(f"   Endpoint: {GPU_BASE_URL}")
    print(f"   Model:    {GPU_MODEL}")

GPU_API_KEY = 6da0856ebacbc687c811e4f86b29f4bb48924838e5e11468eec95f3fe9e34637


In [42]:
# Quick TCP connectivity check before touching Hermes config
# (Catches port-forwarding / firewall / VM-down problems early and with a
# clear diagnosis, instead of waiting ~30-60s for `hermes -z` to time out
# with a generic error further down.)
import socket
from urllib.parse import urlparse

parsed = urlparse(GPU_BASE_URL)
host, port = parsed.hostname, parsed.port or 80
print(f"Testing TCP connection to {host}:{port} ...")
try:
    sock = socket.create_connection((host, port), timeout=8)
    sock.close()
    print(f"✅ Reachable: {host}:{port}")
except Exception as ex:
    print(f"❌ Could not reach {host}:{port} — {ex}")
    print("   Check: is the vLLM container running on the L4 VM? Is the port forwarded?")
    print("   Any firewall/security-group rule needed? (see the GPU VM exposure setup doc)")
    print("   The next cells will likely time out until this is reachable.")

Testing TCP connection to 103.42.51.73:1027 ...
✅ Reachable: 103.42.51.73:1027


In [43]:
# Reconfigure Hermes to use the L4 GPU model instead of local Ollama
#
# NOTE: this uses a NAMED provider entry under "providers:", not the inline
# "model: {provider: custom, base_url: ...}" shorthand. That inline path has a
# confirmed, open Hermes Agent bug (github.com/NousResearch/hermes-agent/issues/43586):
# it never reads api_key or key_env from the model block at all, and silently
# sends "Authorization: Bearer no-key-required" - which produces an HTTP 401 even
# with a correct key. The named-provider path does not have this bug.
import os, yaml

config_path = os.path.expanduser("~/.hermes/config.yaml")
env_path = os.path.expanduser("~/.hermes/.env")

with open(config_path, "r") as f:
    config = yaml.safe_load(f) or {}

# Save the old (local Ollama) model so it's easy to see what changed / switch back
old_model = config.get("model", "unknown")

PROVIDER_NAME = "l4vllm"

config["model"] = {
    "default": GPU_MODEL,
    "provider": PROVIDER_NAME,
    "context_length": 65536,
}

if "providers" not in config:
    config["providers"] = {}

config["providers"][PROVIDER_NAME] = {
    "name": PROVIDER_NAME,
    "base_url": GPU_BASE_URL,
    "key_env": "GPU_API_KEY",
    "default_model": GPU_MODEL,
    "api_mode": "chat_completions",
}

with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

# key_env points at this variable name - it must exist in .env
env_vars = {}
if os.path.exists(env_path):
    with open(env_path, "r") as f:
        for line in f:
            line = line.strip()
            if "=" in line and not line.startswith("#"):
                k, v = line.split("=", 1)
                env_vars[k.strip()] = v.strip()

env_vars["GPU_API_KEY"] = GPU_API_KEY

with open(env_path, "w") as f:
    for k, v in env_vars.items():
        f.write(f"{k}={v}\n")

print("✅ Hermes reconfigured")
print(f"   Was: {old_model}")
print(f"   Now: {config['model']} -> provider '{PROVIDER_NAME}'")

✅ Hermes reconfigured
   Was: {'context_length': 64000, 'default': 'llama3.2:1b', 'provider': 'localollama'}
   Now: {'default': 'hermes-3-llama-3.1-8b', 'provider': 'l4vllm', 'context_length': 65536} -> provider 'l4vllm'


In [44]:
# Quick sanity check: confirm Hermes can reach the L4 GPU model
import subprocess, time

SANITY_TIMEOUT_S = 60  # generous - cold-start + first-inference latency can exceed 30s

print(f"Sending a test prompt to {GPU_MODEL} on Neysa L4 GPU (timeout: {SANITY_TIMEOUT_S}s)...")
start = time.time()
try:
    result = subprocess.run(
        ["hermes", "-z", "Say exactly: 'L4 GPU connection successful!' and nothing else."],
        capture_output=True, text=True, timeout=SANITY_TIMEOUT_S, env=os.environ
    )
    elapsed = time.time() - start
    response = result.stdout or result.stderr
    print(f"Response ({elapsed:.1f}s):", response.strip())

    if "successful" in response.lower():
        print(f"\n🚀 Connected to {GPU_MODEL} on your L4 GPU!")
    else:
        print("\n⚠️  Unexpected response — check that the vLLM server is running, the API key")
        print("   matches, and the port is reachable (see the GPU VM exposure setup doc).")
except subprocess.TimeoutExpired:
    elapsed = time.time() - start
    print(f"\n❌ No response after {elapsed:.0f}s — Hermes never got a reply from the L4 GPU endpoint.")
    print("   This is NOT the same as the HTTP 401 error from before - a 401 comes back")
    print("   almost instantly. A full timeout usually means one of:")
    print("     1. The vLLM server process on the L4 VM has stopped or crashed")
    print("        (SSH in and check: docker ps / docker logs <container>)")
    print("     2. The GPU is still loading the model into memory (cold start after a")
    print("        VM restart) - wait a minute and re-run this cell")
    print("     3. A firewall/security-group rule is silently dropping packets instead")
    print("        of actively refusing them (the TCP check above would normally catch")
    print("        this - if it passed but this still times out, the issue is likely")
    print("        vLLM itself hanging, not the network)")
    print("   Re-run the TCP connectivity check cell above for a faster diagnosis.")

Sending a test prompt to hermes-3-llama-3.1-8b on Neysa L4 GPU (timeout: 60s)...
Response (6.4s): L4 GPU connection successful!

🚀 Connected to hermes-3-llama-3.1-8b on your L4 GPU!


In [45]:
# Test the code reveiewer skill — ask Hermes to review some buggy code

buggy_code = '''
def calculate_average(numbers):
    total = 0
    for n in numbers:
        total = total + n
    average = total / len(numbers)  # BUG: what if numbers is empty?
    return average
    
password = "admin123"  # BUG: hardcoded secret
'''

prompt = f"/code-reviewer Please review this Python code:\n```python{buggy_code}```"

result = subprocess.run(
    ["hermes", "-z", prompt],
    capture_output=True, text=True, timeout=120, env=os.environ
)

print("=== SKILL RESPONSE ===")
print(result.stdout or result.stderr)

=== SKILL RESPONSE ===
Here are a few issues I noticed in the provided Python code:

1. The `calculate_average` function does not handle the case when the input list `numbers` is empty. If `numbers` is an empty list, dividing by `len(numbers)` will raise a `ZeroDivisionError`. To fix this, you can add a check to ensure that `numbers` is not empty before performing the division. Here's an example:

```python
def calculate_average(numbers):
    if not numbers:  # Check if numbers is empty
        return None  # Return None or handle empty list case
    total = 0
    for n in numbers:
        total = total + n
    average = total / len(numbers)
    return average
```

2. The code contains a hardcoded secret, the `password` variable with the value `"admin123"`. Hardcoded secrets are a security risk and should be avoided. Instead, consider storing sensitive information like passwords in a secure configuration file or using environment variables. You can then retrieve the value from the conf

---
## Part 5 — Use-Case: Gmail Spam Detector

### Overview

We'll build a Hermes skill that:
1. Connects to your Gmail inbox via IMAP
2. Reads incoming emails
3. Classifies each as **SPAM** or **NOT SPAM** with reasoning
4. Optionally marks spam and moves it to the Spam folder

### Two modes:
- **Demo mode** (no Gmail needed): Pass email text directly to Hermes
- **Live mode** (requires Gmail App Password): Connect to real inbox

This is also where we leave the local MacBook Air model behind. Parts 1–3 showed you what a 1B model running on your own laptop can (and can't) do — for this real use-case, we point Hermes at a properly-sized 8B model running on an L4 GPU instead.

### Step 5a: Create the Spam Detector Skill

In [46]:
import os

spam_skill_dir = os.path.join(skills_dir, "spam-detector")
os.makedirs(spam_skill_dir, exist_ok=True)

spam_skill = """---
name: spam-detector
description: Analyze emails and classify them as SPAM or NOT SPAM with detailed reasoning
---

# Spam Detector Skill

You are an expert email spam classifier. You analyze email content with high precision,
understanding both obvious spam and sophisticated phishing attempts.

## Spam Signals to Check

### Strong Spam Indicators
- Urgent language: "ACT NOW", "LIMITED TIME", "Your account will be closed"
- Unsolicited offers: lottery wins, inheritance, job offers you never applied for
- Suspicious senders: mismatched domain names, random characters in address
- Requests for personal info: passwords, SSN, bank details, OTPs
- Malformed URLs: tiny URLs hiding destinations, lookalike domains (paypa1.com)
- Grammar/spelling errors combined with monetary promises
- Excessive capitalization, multiple exclamation marks

### Legitimate Email Indicators  
- Sender domain matches company name
- Personalized content (uses your real name, account details)
- Expected communication (receipt for a purchase you made)
- Professional tone and correct grammar
- Clear unsubscribe option (for newsletters)
- Consistent branding

## Classification Process

1. Examine the sender address and display name
2. Read the subject line for red flags
3. Analyze the body for spam signals
4. Check any links mentioned (without clicking)
5. Assess overall context and plausibility
6. Assign a spam probability score (0-100%)

## Output Format

For each email, respond with:

```
📧 EMAIL ANALYSIS
─────────────────
Verdict:       SPAM | NOT SPAM | SUSPICIOUS
Confidence:    XX%
Category:      [Phishing / Marketing / Scam / Newsletter / Legitimate / etc.]

🚩 Red Flags Found:
  • [list each red flag]

✅ Legitimate Signals:
  • [list any legitimate signals]

📝 Reasoning:
  [2-3 sentence explanation]

🎯 Recommended Action:
  [Delete / Move to Spam / Unsubscribe / Safe to read / Report phishing]
```

## Rules

- Never click or visit URLs in emails
- When uncertain, classify as SUSPICIOUS and explain why
- For phishing attempts, always recommend "Report phishing"
- Provide confidence level for every classification
- If analyzing multiple emails, process them in order and number your responses
"""

with open(os.path.join(spam_skill_dir, "SKILL.md"), "w") as f:
    f.write(spam_skill)

print("✅ Spam Detector skill created!")
print(f"   Location: {spam_skill_dir}/SKILL.md")

✅ Spam Detector skill created!
   Location: /Users/mohanrajdavala/.hermes/skills/spam-detector/SKILL.md


### Step 5b: Demo Mode — Analyze Sample Emails

In [47]:
# Sample emails to classify
sample_emails = [
    {
        "from": "winner@lottery-global-winners2024.com",
        "subject": "YOU WON $1,500,000!!! CLAIM NOW!!!",
        "body": """Dear Lucky Winner,
Congratulations!! Your email was selected in our GLOBAL LOTTERY DRAW.
You have won ONE MILLION FIVE HUNDRED THOUSAND DOLLARS ($1,500,000 USD)!!!
To claim your prize IMMEDIATELY, send us:
- Your full name
- Home address
- Bank account number and routing number
- A processing fee of $199 via Western Union
ACT NOW - offer expires in 24 HOURS!!!
Contact: claim@lottery-payout-now.net"""
    },
    {
        "from": "no-reply@amazon.com",
        "subject": "Your Amazon order #114-8829384-9023847 has shipped",
        "body": """Hello Mohan,
Your order has shipped and is on its way!
Order #114-8829384-9023847
Item: Mechanical Keyboard - Cherry MX Blue
Estimated delivery: June 27, 2026
Track your package: https://www.amazon.com/gp/your-orders
Thank you for shopping with Amazon."""
    },
    {
        "from": "security-alert@paypa1-support.com",
        "subject": "Urgent: Your PayPal account has been limited",
        "body": """Dear PayPal Customer,
We have noticed unusual activity on your account and have temporarily limited access.
To restore full access, please verify your information within 24 hours:
Click here: http://paypa1-verify.secure-login.ru/restore
You will need to provide:
- Email and password
- Credit card number
- SSN for identity verification
Failure to comply will result in permanent account suspension.
PayPal Security Team"""
    }
]

print("✅ 3 sample emails ready for analysis")
for i, email in enumerate(sample_emails, 1):
    print(f"  Email {i}: From: {email['from']}")

✅ 3 sample emails ready for analysis
  Email 1: From: winner@lottery-global-winners2024.com
  Email 2: From: no-reply@amazon.com
  Email 3: From: security-alert@paypa1-support.com


In [48]:
# Build the prompt and send to Hermes with the spam-detector skill

def format_email(email):
    return f"""FROM: {email['from']}
SUBJECT: {email['subject']}
BODY:
{email['body']}"""

emails_text = "\n\n" + "="*50 + "\n\n".join(
    [f"EMAIL {i+1}:\n" + format_email(e) for i, e in enumerate(sample_emails)]
)

prompt = f"/spam-detector Analyze these 3 emails and classify each one:\n{emails_text}"

print(f"Sending to Hermes ({GPU_MODEL} on L4 GPU)...")
print("=" * 60)

result = subprocess.run(
    ["hermes", "-z", prompt],
    capture_output=True, text=True, timeout=180, env=os.environ
)

print(f"=== SPAM DETECTOR RESULTS ({GPU_MODEL} on L4 GPU) ===")
print(result.stdout or result.stderr)

Sending to Hermes (hermes-3-llama-3.1-8b on L4 GPU)...
=== SPAM DETECTOR RESULTS (hermes-3-llama-3.1-8b on L4 GPU) ===
Here are the classifications for the 3 emails:

EMAIL 1 (lottery-global-winners2024.com):
- SPAM. It has all the hallmarks of a lottery scam:
  - Fake lottery win notice 
  - Urgent claim request for personal details and money
  - Suspicious email address (winner@lottery-global-winners2024.com)
  - Poor grammar and formatting
  - No verifiable link to the supposed lottery organization

EMAIL 2 (no-reply@amazon.com):  
- NOT SPAM. It appears to be legitimate:
  - From Amazon (a known entity)
  - Correct formatting and branding
  - Confirms shipping of a real order
  - Includes a tracking link
  - No requests for personal information or money

EMAIL 3 (security-alert@paypa1-support.com):
- SPAM. It is a fake PayPal security alert:
  - Suspicious email address (paypa1-support.com vs paypal.com)
  - Urgent request for personal details and verification
  - Link goes to a su

### Step 5c: Live Mode — Connect to Real Gmail Inbox

> **Skip this section if you don't want to connect to Gmail.**

**One-time setup in your Google Account (do this before running the cells below):**

1. **Turn on 2-Step Verification** → https://myaccount.google.com/security → "2-Step Verification" → follow the prompts. App Passwords (step 3) don't exist until this is on.
2. **Enable IMAP access** → open Gmail in a browser → ⚙️ **Settings** → **See all settings** → **Forwarding and POP/IMAP** tab → under "IMAP access" select **Enable IMAP** → **Save Changes**. Gmail ships with IMAP *off* by default — skipping this is the #1 cause of the `Gmail connection failed` error below.
3. **Create an App Password** → https://myaccount.google.com/apppasswords → App: **Mail**, Device: **Other (Custom name)** → name it e.g. `Hermes` → **Generate**.
4. **Copy the 16-character app password** Google shows you. This is *not* your normal Google password — your normal password will not work here, and Hermes never sees or needs it.

**Then, in the next cell:**
- Set `GMAIL_ADDRESS` to your Gmail address.
- Set `GMAIL_APP_PASSWORD` to the 16-character password from step 4 (spaces in it are fine, the code strips them).
- The app password only grants IMAP mail access — you can revoke it anytime from the same App Passwords page without touching your main account password.

**To actually test spam detection on a live email:**

1. Run the next two cells (credentials, then fetch) to confirm Hermes can pull your `NUM_EMAILS_TO_CHECK` most recent inbox emails.
2. Send yourself a test email — from another address, forward yourself an obvious promo/spam email, or just send a plain test message from your phone.
3. Bump `NUM_EMAILS_TO_CHECK` up if needed so the new email is included in the fetch (it grabs the N most recent), then re-run the fetch cell.
4. Run the final "Analyze live emails" cell and check the `=== LIVE EMAIL SPAM ANALYSIS ===` output for your test email's verdict.

In [1]:
# ============================================================
# 📧 OPTIONAL: Fill in Gmail credentials
# ============================================================

GMAIL_ADDRESS = ".....@gmail.com"
GMAIL_APP_PASSWORD = ".... .... .... ...."
NUM_EMAILS_TO_CHECK = 3

# ============================================================

GMAIL_ENABLED = (
    not GMAIL_ADDRESS.startswith("your.") and 
    not GMAIL_APP_PASSWORD.startswith("xxxx")
)

if GMAIL_ENABLED:
    print("✅ Gmail credentials set — live mode enabled")
else:
    print("ℹ️  Gmail not configured — running in demo mode only")
    print("   Fill in GMAIL_ADDRESS and GMAIL_APP_PASSWORD above to enable live mode")

✅ Gmail credentials set — live mode enabled


In [24]:
# Fetch real emails from Gmail and analyze them

if GMAIL_ENABLED:
    import imaplib
    import email
    import socket
    from email.header import decode_header

    def fetch_recent_emails(address, app_password, count=5):
        """Fetch recent emails from Gmail inbox via IMAP."""
        # Connect to Gmail IMAP (explicit timeout so a blocked network fails fast, not silently)
        mail = imaplib.IMAP4_SSL("imap.gmail.com", 993, timeout=15)
        mail.login(address, app_password.replace(" ", ""))
        mail.select("INBOX")

        # Search for all emails, get most recent 'count'
        _, message_ids = mail.search(None, "ALL")
        ids = message_ids[0].split()
        recent_ids = ids[-count:] if len(ids) >= count else ids

        emails_data = []
        for msg_id in reversed(recent_ids):  # newest first
            _, msg_data = mail.fetch(msg_id, "(RFC822)")
            raw_email = msg_data[0][1]
            parsed = email.message_from_bytes(raw_email)

            # Decode subject
            subject_raw = parsed.get("Subject", "")
            subject_parts = decode_header(subject_raw)
            subject = "".join(
                part.decode(enc or "utf-8") if isinstance(part, bytes) else part
                for part, enc in subject_parts
            )

            # Get body (text/plain part only)
            body = ""
            if parsed.is_multipart():
                for part in parsed.walk():
                    if part.get_content_type() == "text/plain":
                        body = part.get_payload(decode=True).decode(errors="replace")[:800]
                        break
            else:
                body = parsed.get_payload(decode=True).decode(errors="replace")[:800]

            emails_data.append({
                "from": parsed.get("From", ""),
                "subject": subject,
                "body": body
            })

        mail.logout()
        return emails_data

    print(f"Fetching {NUM_EMAILS_TO_CHECK} most recent emails from {GMAIL_ADDRESS}...")
    try:
        live_emails = fetch_recent_emails(GMAIL_ADDRESS, GMAIL_APP_PASSWORD, NUM_EMAILS_TO_CHECK)
        print(f"✅ Fetched {len(live_emails)} emails")
        for i, e in enumerate(live_emails, 1):
            print(f"  {i}. {e['subject'][:60]}... | from: {e['from'][:40]}")
    except (TimeoutError, socket.timeout, ConnectionError, OSError) as ex:
        # These come from the TCP/TLS layer, before login is ever attempted -
        # almost always a NETWORK block, not a bad password or IMAP setting.
        print(f"❌ Gmail connection failed: {ex}")
        print("   This is a NETWORK-level failure (connection/TLS handshake to")
        print("   imap.gmail.com:993 never completed) - it happens before your")
        print("   App Password is even checked. Some networks (corporate/campus")
        print("   Wi-Fi, VPNs, security appliances) block or intercept IMAPS (port 993)")
        print("   traffic while allowing normal HTTPS browsing.")
        print("   Fix: try a different network (e.g. phone hotspot), or use")
        print("   Demo Mode (Step 5c above) instead for this session.")
        GMAIL_ENABLED = False
    except Exception as ex:
        # imaplib.IMAP4.error and similar - connection succeeded, so this is
        # a credentials / account-setting problem.
        print(f"❌ Gmail connection failed: {ex}")
        print("   Check your App Password and that IMAP is enabled in Gmail settings")
        GMAIL_ENABLED = False
else:
    print("⏭️  Skipping Gmail fetch (credentials not set)")

Fetching 3 most recent emails from lazycodingjoker@gmail.com...
❌ Gmail connection failed: _ssl.c:993: The handshake operation timed out
   This is a NETWORK-level failure (connection/TLS handshake to
   imap.gmail.com:993 never completed) - it happens before your
   App Password is even checked. Some networks (corporate/campus
   Wi-Fi, VPNs, security appliances) block or intercept IMAPS (port 993)
   traffic while allowing normal HTTPS browsing.
   Fix: try a different network (e.g. phone hotspot), or use
   Demo Mode (Step 5c above) instead for this session.


In [ ]:
# Analyze live emails with Hermes spam detector

if GMAIL_ENABLED and 'live_emails' in dir():
    emails_text = "\n\n" + "="*50 + "\n\n".join(
        [f"EMAIL {i+1}:\n" + format_email(e) for i, e in enumerate(live_emails)]
    )
    prompt = f"/spam-detector Analyze these {len(live_emails)} real emails from my inbox:\n{emails_text}"
    
    print(f"Analyzing live emails with Hermes ({GPU_MODEL} on L4 GPU)...")
    result = subprocess.run(
        ["hermes", "-z", prompt],
        capture_output=True, text=True, timeout=300, env=os.environ
    )
    print(f"=== LIVE EMAIL SPAM ANALYSIS ({GPU_MODEL} on L4 GPU) ===")
    print(result.stdout or result.stderr)
else:
    print("ℹ️  Using demo emails instead (Gmail not connected)")

---
## Part 6 — Conclusion: Local Laptop vs. GPU-Hosted Model

### What we built:

In [25]:
# List all skills we've created
import os

skills_dir = os.path.expanduser("~/.hermes/skills")
print("📂 Skills installed in ~/.hermes/skills/")
print("="*40)

for skill_name in sorted(os.listdir(skills_dir)):
    skill_path = os.path.join(skills_dir, skill_name, "SKILL.md")
    if os.path.exists(skill_path):
        with open(skill_path, "r") as f:
            content = f.read()
        # Extract description from frontmatter
        desc = ""
        for line in content.split("\n"):
            if line.startswith("description:"):
                desc = line.replace("description:", "").strip()
                break
        print(f"  /{skill_name}")
        print(f"    {desc}")

📂 Skills installed in ~/.hermes/skills/
  /code-reviewer
    Review Python code for bugs, style issues, and suggest improvements
  /computer-use
    |
  /deep-research
    Answer questions using deep research from real peer-reviewed academic papers via MCP research tools
  /dogfood
    "Exploratory QA of web apps: find bugs, evidence, reports."
  /spam-detector
    Analyze emails and classify them as SPAM or NOT SPAM with detailed reasoning
  /yuanbao
    "Yuanbao (元宝) groups: @mention users, query info/members."


# Recap: what changed when we moved from laptop to GPU

========================================================================================================================
## LOCAL (Llama 3.2 1B on your MacBook Air, Parts 1-3)
========================================================================================================================

  1.  Ollama doesn't even list "tool use" as a 1B capability —
      it often described a shell command instead of actually invoking it,
      or hallucinated a plausible-looking answer with no real tool call.
  2.  Struggled with the basic file-listing test in Part 3
  3.  Would likely be similarly unreliable on the spam-detector skill

  1.  Runs entirely offline — no API keys, no GPU, no cloud account
  2.  Zero cost, zero data leaves your laptop
  3.  Great for learning how Hermes and Skills work mechanically

========================================================================================================================
## GPU-HOSTED (Hermes-3-Llama-3.1-8B on your L4 VM, Part 5)
========================================================================================================================
  1.  Reliable tool calling — vLLM's `hermes` tool-call parser matches this
      model's native function-calling format
  2.  8B parameters vs. 1B — meaningfully stronger reasoning and instruction
      following on the spam-detector skill
  3.  Same Hermes config, same Skill file — only the model endpoint changed
  4.  Requires a GPU VM, a running vLLM server, and network exposure/API-key
      management — more moving parts than "just run Ollama locally"

Context: same Hermes install, same spam-detector Skill, same emails — the only thing that changed between Parts 1-3 and Part 5 was which model Hermes was pointed at. That's the point: Skills are portable, model quality is not.

👉 Next: Notebook 2 goes further still, running an even larger model for
   the complex task — see how much more headroom is left.
========================================================================================================================

---
## Summary

You've successfully:

1. ✅ **Installed Hermes Agent** from scratch
2. ✅ **Installed Ollama and ran a local LLM** (Llama 3.2 1B — no API keys, 128K context, runs on a MacBook Air)
3. ✅ **Learned the Skills format** — simple Markdown files that teach Hermes
4. ✅ **Saw the limits** of a small local model firsthand (Part 3)
5. ✅ **Reconfigured Hermes** to point at an 8B model on an L4 GPU — same Skill, no code changes
6. ✅ **Built a Gmail Spam Detector** skill and tested it against your live inbox, running on the GPU model

### Key Takeaway

Skills are just **Markdown files**. You don't need to code. You write instructions, and Hermes follows them. The quality of the output depends heavily on the underlying model — and that gap is easiest to feel by starting on the smallest model that still runs.

---

**→ Continue to Notebook 2: `02_hermes_neysa_velocis.ipynb`** to see Hermes connected to Qwen2.5-32B-Instruct on an H100 GPU, plus a real research-paper MCP tool for agentic deep research.

---
### Resources
- Hermes Agent Docs: https://hermes-agent.nousresearch.com/docs/
- Ollama Model Library: https://ollama.com/library
- Skills Hub: https://agentskills.io